<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/Savings_Assistant_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pseudocode

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [27]:
!pip install gradio pandas hands-on-ai

# Import core libraries
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("📦 Core libraries loaded successfully!")
print(f"Pandas version: {pd.__version__}")

📦 Core libraries loaded successfully!
Pandas version: 2.2.2


In [30]:
from hands_on_ai.chat import get_response

# Test the connection to the hands-on-ai server
try:
    response = get_response("Hello! I'm building a Smart Finance Assistant.")
    print("✅ Hands-on-AI connection successful!")
    print(f"Response: {response}")
except Exception as e:
    print(f"❌ Connection issue: {e}")
    print("You can still work on the data processing foundation without this.")

🧠 Loading model 'llama3.2' into RAM... give me a sec...


[WARNING] Error during request (attempt 1): Connection error.


Oops! I got a little tangled up... Let's try that again 😊


[WARNING] Error during request (attempt 2): Connection error.


✅ Hands-on-AI connection successful!
Response: ❌ Error: Connection error.


In [28]:
import os
from getpass import getpass

# Configure hands-on-ai server connection
os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = getpass('Enter your API key: ')

print("🔑 Hands-on-AI configured successfully!")

Enter your API key: ··········
🔑 Hands-on-AI configured successfully!


In [ ]:
# Initialize a global dictionary to hold the user's financial profile
user_profile = {
    "name": "",
    "monthly_salary": 0.0,
    "emergency_goal_monthly": 0.0,
    "has_secondary_goal": False,
    "secondary_goal_amount": 0.0,
    "secondary_goal_date": "",
    "total_expenses": 0.0,
    "calculated_savings_capacity": 0.0
}

In [24]:
def process_user_profile(name, salary, emergency_amount, use_secondary, secondary_amount, secondary_date):
    global user_profile

    # Clean up name input
    if not name or name.strip() == "":
        return "⚠️ Validation Error: Please enter your name to set up your profile."

    # 1. Validate Salary (Must be strictly greater than 0)
    try:
        val_salary = float(salary) if salary else 0.0
        if val_salary <= 0:
            return "⚠️ Validation Error: Monthly salary must be a positive number greater than $0.00."
    except ValueError:
        return "⚠️ Validation Error: Please enter a valid number for monthly salary."

    # 2. Validate Emergency Target (Must be strictly greater than 0)
    try:
        val_emergency = float(emergency_amount) if emergency_amount else 0.0
        if val_emergency <= 0:
            return "⚠️ Validation Error: Emergency savings goal must be a positive number greater than $0.00."
    except ValueError:
        return "⚠️ Validation Error: Please enter a valid number for your emergency savings goal."

    # 3. Validate Secondary Goal (If selected, amount must be strictly greater than 0)
    val_secondary = 0.0
    if use_secondary:
        try:
            val_secondary = float(secondary_amount) if secondary_amount else 0.0
            if val_secondary <= 0:
                return "⚠️ Validation Error: Secondary savings goal amount must be a positive number greater than $0.00."
            if not secondary_date or secondary_date.strip() == "":
                return "⚠️ Validation Error: Please provide a target date for your secondary goal."
        except ValueError:
            return "⚠️ Validation Error: Please enter a valid number for your secondary goal amount."

    # If all checks pass, save values into our dictionary memory
    user_profile["name"] = name.strip()
    user_profile["monthly_salary"] = val_salary
    user_profile["emergency_goal_monthly"] = val_emergency
    user_profile["has_secondary_goal"] = use_secondary
    user_profile["secondary_goal_amount"] = val_secondary
    user_profile["secondary_date"] = secondary_date

    # Friendly confirmation message for the UI
    confirmation_msg = f"### 🎉 Profile Saved Successfully!\n" \
                       f"Welcome **{user_profile['name']}**! Your monthly salary of **${user_profile['monthly_salary']:,.2f}** " \
                       f"and emergency savings goal of **${user_profile['emergency_goal_monthly']:,.2f}/month** have been safely recorded."

    if use_secondary:
        confirmation_msg += f"\n\nWe are also tracking your secondary goal of **${user_profile['secondary_goal_amount']:,.2f}** by **{secondary_date}**."

    return confirmation_msg

In [25]:
import pandas as pd
from datetime import datetime

def calculate_savings(manual_expenses, csv_file):
    global user_profile

    # First, check if a profile exists by testing salary
    if user_profile["monthly_salary"] <= 0:
        return "⚠️ System Error: No active user profile found. Please set up and save your profile in Tab 1 first!"

    # 1. Determine total expenses and apply constraints
    if csv_file is not None:
        try:
            df = pd.read_csv(csv_file.name)
            if 'Amount' in df.columns:
                total_expenses = df['Amount'].abs().sum()
            elif 'Cost' in df.columns:
                total_expenses = df['Cost'].abs().sum()
            else:
                numeric_cols = df.select_dtypes(include=['number']).columns
                if len(numeric_cols) > 0:
                    total_expenses = df[numeric_cols[0]].abs().sum()
                else:
                    return "⚠️ CSV Error: Could not find any numeric expense column in your file."

            # Constraint for empty or zero-value CSV data
            if total_expenses <= 0:
                return "⚠️ CSV Error: The uploaded file contains no transactions or total expenses equal $0.00."

        except Exception as e:
            return f"⚠️ Error processing CSV file: {str(e)}"
    else:
        # Validate manual input constraints
        try:
            total_expenses = float(manual_expenses) if manual_expenses else 0.0
            if total_expenses <= 0:
                return "⚠️ Validation Error: Monthly expenses must be a positive number greater than $0.00."
        except ValueError:
            return "⚠️ Validation Error: Please enter a valid number for your monthly expenses."

    # Save total expenses to our memory dictionary
    user_profile["total_expenses"] = total_expenses

    # 2. Financial Math Calculations
    salary = user_profile["monthly_salary"]
    leftover_income = salary - total_expenses
    user_profile["calculated_savings_capacity"] = leftover_income

    emergency_target = user_profile["emergency_goal_monthly"]
    total_needed_monthly = emergency_target

    secondary_details = ""
    if user_profile["has_secondary_goal"] and user_profile["secondary_goal_amount"] > 0:
        sec_amount = user_profile["secondary_goal_amount"]
        sec_date_str = user_profile["secondary_date"]

        months_remaining = 1
        try:
            target_date = datetime.strptime(sec_date_str, "%Y-%m-%d")
            current_date = datetime.now()
            months_remaining = (target_date.year - current_date.year) * 12 + target_date.month - current_date.month
            if months_remaining <= 0:
                months_remaining = 1
        except:
            pass

        sec_monthly_needed = sec_amount / months_remaining
        total_needed_monthly += sec_monthly_needed
        secondary_details = f"* **Secondary Goal Monthly Target:** ${sec_monthly_needed:,.2f} (To reach ${sec_amount:,.2f} by {sec_date_str})\n"

    # 3. Generate Report String
    report = f"## 📊 Financial Analysis Report for {user_profile['name']}\n" \
             f"---\n" \
             f"* **Total Income:** ${salary:,.2f}\n" \
             f"* **Total Logged Expenses:** ${total_expenses:,.2f}\n" \
             f"* **Remaining Cash Flow:** ${leftover_income:,.2f}\n\n" \
             f"### 🎯 Savings Commitments:\n" \
             f"* **Emergency Fund Target:** ${emergency_target:,.2f}/month\n" \
             f"{secondary_details}" \
             f"* **Total Ideal Monthly Savings:** ${total_needed_monthly:,.2f}\n" \
             f"---\n"

    if leftover_income >= total_needed_monthly:
        report += f"### ✅ Status: On Track!\n" \
                  f"Excellent job! You have enough cash flow to cover your goals and still have " \
                  f"**${leftover_income - total_needed_monthly:,.2f}** remaining for leisurely spending."
    elif leftover_income >= emergency_target:
        report += f"### ⚠️ Status: Partial Progress\n" \
                  f"You can comfortably cover your Emergency Fund, but you are short by " \
                  f"**${total_needed_monthly - leftover_income:,.2f}** to hit your secondary goal target simultaneously. Consider reducing non-essential spending."
    else:
        report += f"### ❌ Status: Budget Deficit\n" \
                  f"Warning! Your current monthly expenses do not leave you enough room to hit your baseline savings goals. " \
                  f"You are short by **${total_needed_monthly - leftover_income:,.2f}** this month. Ask your Chat Coach for budgeting tips!"

    return report

In [32]:
from hands_on_ai.chat import get_response # Corrected import

def chat_with_savings_coach(user_message, chat_history):
    global user_profile

    # 1. Fetch current live data from our system memory
    name = user_profile["name"] if user_profile["name"] else "Savings Buddy User"
    salary = user_profile["monthly_salary"]
    expenses = user_profile["total_expenses"]
    capacity = user_profile["calculated_savings_capacity"]

    # 2. Add Business Framing: Calculate the expenditure ratio
    expense_ratio = (expenses / salary) * 100 if salary > 0 else 0

    # 3. Create a powerful System Prompt giving the AI its persona and data context
    system_prompt = (
        f"You are 'Savings Buddy', an encouraging, insight-driven personal financial coach. "
        f"Your target audience consists of regular people looking for structured paths to financial security. "
        f"You are talking to {name}. Here is their current verified financial data:\n"
        f"- Monthly Net Salary: ${salary:,.2f}\n"
        f"- Current Logged Monthly Expenses: ${expenses:,.2f} ({expense_ratio:.1f}% of their total income)\n"
        f"- Remaining Available Cash Flow: ${capacity:,.2f}\n\n"
        f"CRITICAL STRATEGIC FRAMEWORK:\n"
        f"Evaluate their budget relative to corporate personal finance benchmarks like the 50/30/20 rule "
        f"(50% Needs, 30% Wants, 20% Financial Savings Goals). "
        f"If their expense ratio is over 50-60%, gently guide them on identifying variable overheads to compress. "
        f"Keep your responses motivating, highly practical, professional, and limited to 2-3 concise paragraphs. "
        f"Do not invent data outside what is provided."
    )

    # 4. Use the university package to call the model with context
    # Note: Adjust the exact function name if your university starter code uses a different method (e.g., llm.chat)
    try:
        ai_response = get_response( # Corrected function call
            prompt=user_message,
            system_message=system_prompt,
            history=chat_history
        )
        return ai_response
    except Exception as e:
        return f"Coach is adjusting their notes! (Error connecting to AI: {str(e)})"


In [33]:
# 1. Install the official Google GenAI library (if not already installed)
!pip install google-genai -q

import os
from google import genai
from google.genai import types

def chat_with_savings_coach(user_message, chat_history):
    global user_profile

    # 2. Safely initialize the client
    # In Colab, you can set your API key in the left-side 'Secrets' tab (🔑 icon) as GEMINI_API_KEY
    # Or pass it directly: client = genai.Client(api_key="YOUR_KEY")
    try:
        client = genai.Client()
    except Exception:
        # Fallback if no environment key is found yet during setup
        return "⚠️ Setup Note: Please configure your Gemini API Key in Colab Secrets to activate the Coach."

    # 3. Fetch live data from our system memory dictionary
    name = user_profile.get("name", "Savings Buddy User")
    salary = user_profile.get("monthly_salary", 0.0)
    expenses = user_profile.get("total_expenses", 0.0)
    capacity = user_profile.get("calculated_savings_capacity", 0.0)

    # 4. Calculate the financial metrics for Business Framing
    expense_ratio = (expenses / salary) * 100 if salary > 0 else 0

    # 5. Build a powerful system prompt to give the AI its professional persona
    system_instruction = (
        f"You are 'Savings Buddy', an encouraging, insight-driven personal financial coach. "
        f"Your target audience consists of regular people looking for structured paths to financial security. "
        f"You are currently talking to {name}.\n\n"
        f"VERIFIED FINANCIAL DATA FOR THIS USER:\n"
        f"- Monthly Net Salary: ${salary:,.2f}\n"
        f"- Current Logged Monthly Expenses: ${expenses:,.2f} ({expense_ratio:.1f}% of their total income)\n"
        f"- Remaining Available Cash Flow: ${capacity:,.2f}\n\n"
        f"CRITICAL STRATEGIC BUSINESS FRAMEWORK:\n"
        f"Evaluate their budget relative to corporate personal finance benchmarks like the 50/30/20 rule "
        f"(50% Needs, 30% Wants, 20% Financial Savings Goals). "
        f"If their expense ratio is over 50%, gently guide them on identifying variable overheads to compress. "
        f"Keep your responses motivating, highly practical, and limited to 2-3 concise paragraphs. "
        f"Do not invent data outside what is provided in this prompt."
    )

    # 6. Format the chat history into the structure the Google API expects
    formatted_contents = []
    for turn in chat_history:
        # turn[0] is user message, turn[1] is AI response
        formatted_contents.append(types.Content(role="user", parts=[types.Part.from_text(text=turn[0])]))
        formatted_contents.append(types.Content(role="model", parts=[types.Part.from_text(text=turn[1])]))

    # Append the newest message from the user
    formatted_contents.append(types.Content(role="user", parts=[types.Part.from_text(text=user_message)]))

    # 7. Call the Gemini Model (using the recommended gemini-2.5-flash for speedy UI apps)
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=formatted_contents,
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.7
            )
        )
        return response.text
    except Exception as e:
        return f"Coach is adjusting their notes! (Error: {str(e)})"

In [ ]:
import gradio as gr

# Ensure a preset, elegant, clean theme is used (Soft theme is perfect for finance)
with gr.Blocks(theme=gr.themes.Soft(), title="Smart Finance Assistant: Savings Buddy") as app:

    gr.Markdown("# 🏦 Smart Finance Assistant: Savings Buddy")
    gr.Markdown("Welcome to your personal financial cockpit. Set your financial baseline, track your expenditures, and consult with our AI Coach.")

    # TAB 1: USER PROFILE INITIAL SETUP
    with gr.Tab("1. Profile Setup"):
        gr.Markdown("### 👤 Establish Your Financial Baseline")
        with gr.Row():
            name_input = gr.Textbox(label="Your Name", placeholder="e.g., Alex")
            salary_input = gr.Number(label="Monthly Net Salary ($)", value=0.0)
            emergency_input = gr.Number(label="Monthly Emergency Fund Target ($)", value=0.0)

        with gr.Row():
            has_sec_checkbox = gr.Checkbox(label="Do you have a secondary savings goal?", value=False)
            sec_amount_input = gr.Number(label="Secondary Goal Total Target ($)", value=0.0, visible=True)
            sec_date_input = gr.Textbox(label="Target Date (YYYY-MM-DD)", placeholder="e.g., 2027-05-25", visible=True)

        profile_save_btn = gr.Button("💾 Save Profile Data", variant="primary")
        profile_status_output = gr.Markdown("*Profile not yet configured.*")

        # Link profile button to the Python validation function
        profile_save_btn.click(
            fn=process_user_profile,
            inputs=[name_input, salary_input, emergency_input, has_sec_checkbox, sec_amount_input, sec_date_input],
            outputs=profile_status_output
        )

    # TAB 2: EXPENSE TRACKER, CSV PARSER & CALCULATOR
    with gr.Tab("2. Track & Calculate"):
        gr.Markdown("### 📊 Log Monthly Expenditures")
        gr.Markdown("Input your expenses manually *or* upload a financial transaction CSV file containing an 'Amount' or 'Cost' column.")

        with gr.Row():
            manual_expense_input = gr.Number(label="Enter Manual Expenses Total ($)", value=0.0)
            csv_file_input = gr.File(label="Upload Expense CSV File", file_types=[".csv"])

        with gr.Row():
            calculate_btn = gr.Button("📈 Calculate My Savings Capacity", variant="primary")
            reset_btn = gr.Button("🔄 Clear & Reset Expenses", variant="secondary")

        analysis_report_output = gr.Markdown("### 📋 Analysis Summary\n*Please input your expenses and hit calculate to generate a corporate-framed financial statement.*")

        # Link buttons to their respective logic functions
        calculate_btn.click(
            fn=calculate_savings,
            inputs=[manual_expense_input, csv_file_input],
            outputs=analysis_report_output
        )

        reset_btn.click(
            fn=reset_expense_tracker,
            inputs=[],
            outputs=[csv_file_input, manual_expense_input, analysis_report_output]
        )

    # TAB 3: THE AI SAVINGS COACH
    with gr.Tab("3. Ask your Savings Buddy"):
        gr.Markdown("### 🤖 Strategic AI Financial Coach")
        gr.Markdown("Our AI Coach evaluates your cash flow using the 50/30/20 framework to suggest operational expense cutbacks.")

        # Gradio's standard built-in interactive Chat Interface
        chatbot_ui = gr.ChatInterface(
            fn=chat_with_savings_coach,
            type="messages" # Conforms to standard chat array structures
        )

# Launch the app with a share link so you can preview it in your browser immediately
app.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://541e36cfc248a03dfa.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
